In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import pandas as pd
import keras
from keras.utils import FeatureSpace
import numpy as np

I0000 00:00:1773407298.145686   52350 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [3]:
class Keras_Binary_Classification(object):

    def __init__(self, dataset):
        self.dataset = dataset.copy().reset_index(drop = True)

    def Data_Split(self):
        global train_data_global
        global test_data_global
        
        test_data = self.dataset.sample(frac = 0.2, random_state=1337)
        train_data = self.dataset.drop(test_data.index)

        print("Using %d samples for training and %d for validation" % (len(train_data), len(test_data)))

        train_data_global = train_data
        test_data_global = test_data

    def dataframe_to_dataset(self, dataset):
        global ds_global

        df = self.dataset.copy()
        labels = df.pop('HeartDiseaseorAttack')
        ds = tf.data.Dataset.from_tensor_slices((dict(df), labels))
        ds = ds.shuffle(buffer_size=len(df))
        ds_global = ds
        return ds
        
    def batching(self):
        global train_ds_global
        global test_ds_global

        train_ds = self.dataframe_to_dataset(train_data_global)
        train_ds = train_ds.batch(32)
        train_ds_global = train_ds
        test_ds = self.dataframe_to_dataset(test_data_global)
        test_ds = test_ds.batch(32)
        test_ds_global = test_ds

    def FeatureSpace(self):
        global train_ds_with_no_labels_global
        global feature_space_global
        
        feature_space = FeatureSpace(
    features={
        # Categorical features encoded as integers
        "HighBP": "integer_categorical",
        "HighChol": "integer_categorical",
        "CholCheck": "integer_categorical",
        "Smoker": "integer_categorical",
        "Stroke": "integer_categorical",
        "Diabetes": "integer_categorical",
        "PhysActivity": "integer_categorical",
        "Fruits": "integer_categorical",
        "Veggies": "integer_categorical",
        "HvyAlcoholConsump": "integer_categorical",
        "AnyHealthcare": "integer_categorical",
        "NoDocbcCost": "integer_categorical",
        "GenHlth": "integer_categorical",
        "MentHlth": "integer_categorical",
        "PhysHlth": "integer_categorical",
        "DiffWalk": "integer_categorical",
        "Sex": "integer_categorical",
        "Education": "integer_categorical",
        "Income": "integer_categorical",

        # Numerical features to discretize
        "Age": "float_discretized",
        # Numerical features to normalize
        "BMI": "float_normalized",
    },
    # We create additional features by hashing
    # value co-occurrences for the
    # following groups of categorical features.
    crosses=[("Sex", "Age"), ("PhysActivity", "MentHlth")],
    # The hashing space for these co-occurrences
    # wil be 32-dimensional.
    crossing_dim=32,
    # Our utility will one-hot encode all categorical
    # features and concat all features into a single
    # vector (one vector per sample).
    output_mode="concat",
)
        
        train_ds_with_no_labels = train_ds_global.map(lambda x, _: x)
        feature_space.adapt(train_ds_with_no_labels)
        train_ds_with_no_labels_global = train_ds_with_no_labels

        feature_space_global = feature_space

            
    def preprocessing(self):
        global preprocessed_x_global
        global preprocessed_train_ds_global
        global preprocessed_test_ds_global

        for x, _ in train_ds_global.take(1):
            preprocessed_x = feature_space_global(x)
            preprocessed_x_global = preprocessed_x
            print("preprocessed_x.shape:", preprocessed_x.shape)
            print("preprocessed_x.dtype:", preprocessed_x.dtype)

        preprocessed_train_ds = train_ds_global.map(lambda x, y: (feature_space_global(x), y), num_parallel_calls=tf.data.AUTOTUNE)
        preprocessed_train_ds = preprocessed_train_ds.prefetch(tf.data.AUTOTUNE)
        preprocessed_train_ds_global = preprocessed_train_ds

        preprocessed_test_ds = test_ds_global.map(lambda x, y: (feature_space_global(x), y), num_parallel_calls=tf.data.AUTOTUNE)
        preprocessed_test_ds = preprocessed_test_ds.prefetch(tf.data.AUTOTUNE)
        preprocessed_test_ds_global = preprocessed_test_ds

    def model_build(self):
        global inference_model_global
        global training_model_global
        
        dict_inputs = feature_space_global.get_inputs()
        encoded_features = feature_space_global.get_encoded_features()

        x = tf.keras.layers.Dense(32, activation="relu")(encoded_features)
        x = tf.keras.layers.Dropout(0.5)(x)
        predictions = tf.keras.layers.Dense(1, activation="sigmoid")(x)

        training_model = tf.keras.Model(inputs=encoded_features, outputs=predictions)
        training_model.compile(
            optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        training_model_global = training_model

        inference_model = tf.keras.Model(inputs=dict_inputs, outputs=predictions)
        inference_model_global = inference_model

    def model_train(self):
        training_model_global.fit(preprocessed_train_ds_global,
                           epochs=20,
                           validation_data=preprocessed_test_ds_global,
                           verbose=2,)

    def main(self):
        
        self.Data_Split()
        self.batching()
        self.FeatureSpace()
        self.preprocessing()
        self.model_build()
        self.model_train()
        



In [4]:
keras = Keras_Binary_Classification(df)

In [5]:
keras.main()

Using 202944 samples for training and 50736 for validation


I0000 00:00:1773407300.651737   52350 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8861 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:08:00.0, compute capability: 7.5


preprocessed_x.shape: (32, 226)
preprocessed_x.dtype: <dtype: 'float32'>
Epoch 1/20


I0000 00:00:1773407512.837586   52521 service.cc:153] XLA service 0x57d70b10d730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773407512.837606   52521 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.1.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.15.1)
I0000 00:00:1773407512.851623   52521 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773407512.945507   52521 cuda_dnn.cc:461] Loaded cuDNN version 91501
I0000 00:00:1773407512.968763   52521 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1590130__.9
I0000 00:00:1773407513.438697   52521 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1773407520.101577   52519 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1590130__

7928/7928 - 18s - 2ms/step - accuracy: 0.9069 - loss: 0.2474 - val_accuracy: 0.9079 - val_loss: 0.2370
Epoch 2/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9072 - loss: 0.2425 - val_accuracy: 0.9081 - val_loss: 0.2364
Epoch 3/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9074 - loss: 0.2412 - val_accuracy: 0.9082 - val_loss: 0.2362
Epoch 4/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9075 - loss: 0.2407 - val_accuracy: 0.9081 - val_loss: 0.2363
Epoch 5/20
7928/7928 - 15s - 2ms/step - accuracy: 0.9073 - loss: 0.2404 - val_accuracy: 0.9079 - val_loss: 0.2361
Epoch 6/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9074 - loss: 0.2403 - val_accuracy: 0.9082 - val_loss: 0.2370
Epoch 7/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9071 - loss: 0.2402 - val_accuracy: 0.9081 - val_loss: 0.2372
Epoch 8/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9073 - loss: 0.2401 - val_accuracy: 0.9081 - val_loss: 0.2370
Epoch 9/20
7928/7928 - 16s - 2ms/step - accuracy: 0.9074 - loss: 0.2401 - val_accuracy: 0.9085 - va

In [6]:
class Data_Generation(object):

    def __init__(self, dataset, size, data_type):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.size = size
        self.data_type = data_type
        self.HighBP_nums = []
        self.HighChol_nums = []
        self.CholCheck_nums = []
        self.Smoker_nums = []
        self.Stroke_nums = []
        self.PhysActivity_nums = []
        self.Fruits_nums = []
        self.Veggies_nums= []
        self.HvyAlcoholConsump_nums = []
        self.AnyHealthcare_nums = []
        self.NoDocbcCost_nums = []
        self.DiffWalk_nums = []
        self.Sex_nums = []
        self.BMI_nums = []
        self.Diabetes_nums = []
        self.GenHlth_nums = []
        self.MentHlth_nums = []
        self.PhysHlth_nums = []
        self.Age_nums = []
        self.Education_nums = []
        self.Income_nums = []
        self.generated_data = []

    def generating_data(self):
        self.HighBP_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HighChol_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.CholCheck_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Smoker_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Stroke_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.PhysActivity_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Fruits_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Veggies_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HvyAlcoholConsump_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.AnyHealthcare_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.NoDocbcCost_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.DiffWalk_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Sex_nums = np.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.BMI_nums = np.random.choice(range(12, 99), size=self.size).astype(self.data_type)
        self.Diabetes_nums = np.random.choice(range(0, 3), size = self.size).astype(self.data_type)
        self.GenHlth_nums = np.random.choice(range(1, 6), size = self.size).astype(self.data_type)
        self.MentHlth_nums = np.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.PhysHlth_nums = np.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.Age_nums = np.random.choice(range(1, 14), size = self.size).astype(self.data_type)
        self.Education_nums = np.random.choice(range(1, 7), size = self.size).astype(self.data_type)
        self.Income_nums = np.random.choice(range(1, 9), size = self.size).astype(self.data_type)
        
        
    def generating_dataframe(self):
        global generated_data_global

        self.generated_data = pd.DataFrame({"HighBP": self.HighBP_nums, 
                                            'HighChol': self.HighChol_nums,
                                            'CholCheck' : self.CholCheck_nums,
                                            'BMI' : self.BMI_nums, 
                                            'Smoker' : self.Smoker_nums, 
                                            'Stroke' : self.Stroke_nums, 
                                            'Diabetes' : self.Diabetes_nums,
                                            'PhysActivity' : self.PhysActivity_nums, 
                                            'Fruits' : self.Fruits_nums, 
                                            'Veggies' : self.Veggies_nums,
                                            'HvyAlcoholConsump' : self.HvyAlcoholConsump_nums, 
                                            'AnyHealthcare' : self.AnyHealthcare_nums, 
                                            'NoDocbcCost' : self.NoDocbcCost_nums,
                                            'GenHlth' : self.GenHlth_nums, 
                                            'MentHlth' : self.MentHlth_nums, 
                                            'PhysHlth' : self.PhysHlth_nums, 
                                            'DiffWalk' : self.DiffWalk_nums, 
                                            'Sex' : self.Sex_nums, 
                                            'Age' : self.Age_nums, 
                                            'Education' : self.Education_nums,
                                            'Income' : self.Income_nums})

        
        generated_data_global = self.generated_data

    def main(self):
        self.generating_data()
        self.generating_dataframe()

In [7]:
gen_df = Data_Generation(df, 1000000, float)

In [8]:
gen_df.main()

In [9]:
input_dict = {name: tf.convert_to_tensor([value]) for name, value in generated_data_global.iloc[15].items()}
predictions = inference_model_global.predict(input_dict)

print(
    f"This particular patient had a {100 * predictions[0][0]:.2f}% probability "
    "of having a heart disease, as evaluated by our model."
)

/home/xy/Desktop/ml/tf/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'Age': 'Age', 'AnyHealthcare': 'AnyHealthcare', 'BMI': 'BMI', 'CholCheck': 'CholCheck', 'Diabetes': 'Diabetes', 'DiffWalk': 'DiffWalk', 'Education': 'Education', 'Fruits': 'Fruits', 'GenHlth': 'GenHlth', 'HighBP': 'HighBP', 'HighChol': 'HighChol', 'HvyAlcoholConsump': 'HvyAlcoholConsump', 'Income': 'Income', 'MentHlth': 'MentHlth', 'NoDocbcCost': 'NoDocbcCost', 'PhysActivity': 'PhysActivity', 'PhysHlth': 'PhysHlth', 'Sex': 'Sex', 'Smoker': 'Smoker', 'Stroke': 'Stroke', 'Veggies': 'Veggies'}
Received: inputs={'HighBP': 'Tensor(shape=(1,))', 'HighChol': 'Tensor(shape=(1,))', 'CholCheck': 'Tensor(shape=(1,))', 'BMI': 'Tensor(shape=(1,))', 'Smoker': 'Tensor(shape=(1,))', 'Stroke': 'Tensor(shape=(1,))', 'Diabetes': 'Tensor(shape=(1,))', 'PhysActivity': 'Tensor(shape=(1,))', 'Fruits': 'Tensor(shape=(1,))', 'Veggies': '

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
This particular patient had a 13.92% probability of having a heart disease, as evaluated by our model.
